In [ ]:
import sys
import os
import subprocess
import numpy as np

# Add project root to path so `src_py` is importable as a package
_root = os.path.normpath(os.path.join(os.path.abspath(''), '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt

from src_py import surrogate_model as srg, muvec as srg_grid
from src_py import surrogate_model_proj as srg_proj


In [ ]:
alpha = 5
gamma = 1.0
phi = 4.0
srg_vcut = srg(alpha, gamma, phi)
srg_proj_vcut = srg_proj(0.0, alpha, gamma, phi)

mu_max = 10
Nmu = 20
srg_proj_grid = np.linspace(0, mu_max, Nmu)

print(f"Surrogate model vcut: {srg_vcut}")
print(f"Surrogate projection vcut: {srg_proj_vcut}")

In [ ]:
figure, ax = plt.subplots(figsize=(5, 3.5))

msg = []
vcut = srg_proj(srg_proj_grid, alpha, gamma, phi, msg=msg)
ax.plot(vcut, srg_proj_grid, marker='o', label='Surrogate Projection')
if srg_vcut is not None:
    ax.plot(srg_vcut, srg_grid, color='r', linestyle='--', label='Surrogate Model')
ax.axvline(x=np.sqrt(phi), color='k', linestyle=':', label=r'$\sqrt{2\phi}$')
ax.set_xlabel(r'$v_{\mathrm{cut}}$')
ax.set_ylabel(r'$\mu$')
ax.set_title(f'Surrogate Projection vs Model (α={alpha}, γ={gamma}, φ={phi})')
ax.legend()
plt.tight_layout()

In [ ]:
from src_py import generate_c_code

generate_c_code(
    nn_model      = "../model/nn_model.pth",
    svm_model     = "../model/svm_model.pkl",
    normalization = "../model/normalization.npz",
    output_dir    = "../generated_c_code",
    gkeyll_dir    = "/Users/ahoffman/gkeyll_main/gkeyll/gyrokinetic/ker/bc_sheath_gyrokinetic",
)

# --- Make clean and compile the C code ---
subprocess.run(["make", "-C", "../generated_c_code", "clean"]);
subprocess.run(["make", "-C", "../generated_c_code"]);

In [ ]:
# --- Run the compiled C test program and capture output ---
proc = subprocess.run(
    ["./../generated_c_code/test_surrogate", "predict"],
    capture_output=True, text=True
)
c_out = np.array([
    float(line.split("=")[1])
    for line in proc.stdout.strip().splitlines()
])

# --- Python reference (same inputs as test_surrogate.c: alpha=4, gamma=1.0, phi=2.5) ---
py_out = srg(4, 1.0, 2.5)

# --- Comparison ---
abs_err = np.abs(c_out - py_out)
print(f"{'i':>3}  {'C output':>14}  {'Python output':>14}  {'abs error':>12}")
print("-" * 50)
for i, (c, p, e) in enumerate(zip(c_out, py_out, abs_err)):
    print(f"{i:>3}  {c:>14.8f}  {p:>14.8f}  {e:>12.2e}")

print(f"\nMax absolute error : {abs_err.max():.2e}")
print(f"Mean absolute error: {abs_err.mean():.2e}")

In [ ]:
# -- Test eval mode ---
alpha = 10.0
gamma = 2.0
phi = 3.0

mugrid_eval = np.linspace(0, 10, 32)

srg_vcut_eval = srg(alpha, gamma, phi)

c_vcut_eval = np.zeros_like(mugrid_eval)
for imu, mu in enumerate(mugrid_eval):
    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "eval", str(mu), str(alpha), str(gamma), str(phi)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vcut_eval[imu] = c_out[0]

plt.figure(figsize=(5, 3.5))
plt.plot(c_vcut_eval, mugrid_eval, marker='o', label='C Eval')
if srg_vcut_eval is not None:
    plt.plot(srg_vcut_eval, srg_grid, color='r', linestyle='--', label='Python Eval')
plt.axvline(np.sqrt(2*phi), color='k', linestyle=':', label=r'$\sqrt{\phi}$')
plt.xlabel(r'$v_{\text{GYRAZE}}$')
plt.ylabel(r'$\mu_{\text{GYRAZE}}$')
plt.legend()

In [ ]:
# -- Test eval physical mode ---
m_e = 9.10938356e-31
epsilon_0 = 8.8541878128e-12
e = 1.602176634e-19
Te = 5 * 1.60218e-19  # 10 eV in Joules
mass = m_e
charge = -e
Bimpact_angle = 5 * np.pi / 180
Bmag = 1.0
dens_e = 1.0e19
phi = 15.0 # V
phi_wall = 0.0 # V
q2Dm = 2 * charge / mass

vt = np.sqrt(Te/mass)

alpha = Bimpact_angle * 180 / np.pi
gamma = np.sqrt(m_e * dens_e / epsilon_0) / Bmag
phinorm = e*(phi - phi_wall) / Te

print(f"{alpha=}, {gamma=}, {phinorm=}")

mugrid_eval = np.linspace(0, 10 * Te / Bmag, 32)

srg_vals = np.zeros_like(mugrid_eval)
c_vals = np.zeros_like(mugrid_eval)

for imu, mu in enumerate(mugrid_eval):
    munorm = mu * Bmag / Te
    srg_vals[imu] = srg_proj(munorm, alpha, gamma, phinorm)

    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "physical", str(mu), str(phi), str(phi_wall), 
         str(dens_e), str(Te), str(Bmag), str(Bimpact_angle)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vals[imu] = c_out[0]

vcut_const = np.sqrt(-q2Dm * (phi - phi_wall)) / vt

plt.figure(figsize=(5, 3.5))
plt.plot(c_vals, mugrid_eval, marker='o', label='C Physical Eval')
plt.plot(srg_vals, mugrid_eval, color='r', linestyle='-', label='Python Physical Eval')
plt.axvline(np.sqrt(phinorm), color='k', linestyle='--', label=r'$\sqrt{\phi_n}$')
plt.axvline(vcut_const, color='k', linestyle=':', label=r'$\sqrt{-2q/m \Delta\phi}/v_{te}$')
plt.xlabel(r'$v_\parallel / v_{te}$')
plt.ylabel(r'$\mu$')
# plt.xlim(0, 2.2)
plt.legend()
plt.show()

In [ ]:
# -- Test eval physical vcut fact mode ---

alpha = Bimpact_angle * 180 / np.pi
gamma = np.sqrt(m_e * dens_e / epsilon_0) / Bmag
phinorm = e*(phi - phi_wall) / Te

print(f"{alpha=}, {gamma=}, {phinorm=}")

mugrid_eval = np.linspace(0, 10 * Te / Bmag, 24)

srg_vals = np.zeros_like(mugrid_eval)
c_vals = np.zeros_like(mugrid_eval)
vcut_const = np.sqrt(phinorm)

for imu, mu in enumerate(mugrid_eval):
    munorm = mu * Bmag / Te
    srg_vals[imu] = srg_proj(munorm, alpha, gamma, phinorm)

    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "physical_vcut_fact", str(mu), str(phi), str(phi_wall), 
         str(dens_e), str(Te), str(q2Dm), str(Bmag), str(Bimpact_angle)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vals[imu] = c_out[0]  # Convert back to physical vcut

# vcut_const = np.sqrt(2*e*(phi-phi_wall)/m_e)/(np.sqrt(Te/m_e))

eta_srgz_python = srg_vals * vt / np.sqrt(-q2Dm * (phi - phi_wall))

plt.figure(figsize=(5, 3.5))
plt.plot(np.sqrt(c_vals), mugrid_eval*Bmag/Te, marker='o', label='C Physical Eval')
plt.plot(eta_srgz_python, mugrid_eval*Bmag/Te, color='r', linestyle='--', label='Python Physical Eval')
plt.axvline(1.0, color='k', linestyle=':', label=r'$v_{c0} = \sqrt{-2q/m \Delta\phi}$')
plt.xlabel(r'$\sqrt{\eta} = v_\parallel / v_{c0}$')
plt.ylabel(r'$\mu B/T$')
plt.legend()
plt.show()

In [ ]:
# -- Test the projector of unconverged parameter set --
# alpha, gamma, phinorm = 5.416269, 0.692704, 0.261429
phinorm, gamma, alpha = 0.00, 0.40, 3.30
Bimpact_angle = alpha * np.pi / 180
dens_e = gamma**2 * Bmag**2 * epsilon_0 / m_e
phi = phinorm * Te / e + phi_wall

print(f"{alpha=}, {gamma=}, {phinorm=}")

mugrid_eval = np.linspace(0, 10 * Te / Bmag, 24)

srg_vals = np.zeros_like(mugrid_eval)
c_vals = np.zeros_like(mugrid_eval)
vcut_const = np.sqrt(phinorm)

for imu, mu in enumerate(mugrid_eval):
    munorm = mu * Bmag / Te
    srg_vals[imu] = srg_proj(munorm, alpha, gamma, phinorm)

    proc_eval = subprocess.run(
        ["./../generated_c_code/test_surrogate", "proj_physical_vcut_fact", str(mu), str(phi), str(phi_wall), 
         str(dens_e), str(Te), str(q2Dm), str(Bmag), str(Bimpact_angle)],
        capture_output=True, text=True
    )
    c_out = np.array([
        float(line.split("=")[1])
        for line in proc_eval.stdout.strip().splitlines()
    ])
    c_vals[imu] = c_out[0]  # Convert back to physical vcut

# vcut_const = np.sqrt(2*e*(phi-phi_wall)/m_e)/(np.sqrt(Te/m_e))
if np.abs(phi - phi_wall) > 0:
    eta_srgz_python = srg_vals * vt / np.sqrt(-q2Dm * (phi - phi_wall))
else:
    eta_srgz_python = np.ones_like(srg_vals)

plt.figure(figsize=(5, 3.5))
plt.plot(np.sqrt(c_vals), mugrid_eval*Bmag/Te, marker='o', label='C Physical Eval')
plt.plot(eta_srgz_python, mugrid_eval*Bmag/Te, color='r', linestyle='--', label='Python Physical Eval')
plt.axvline(1.0, color='k', linestyle=':', label=r'$v_{c0} = \sqrt{-2q/m \Delta\phi}$')
plt.xlabel(r'$\sqrt{\eta} = v_\parallel / v_{c0}$')
plt.ylabel(r'$\mu B/T$')
plt.legend()
plt.show()

print(proc_eval.stderr)

Test the surrogate projection method in C and compare it with python in an unconverged case.

In [ ]:
alpha=4.775999
gamma=0.667020
phi=0.143294

mu = 0.5
msg = []
vcut = srg_proj(mu, alpha, gamma, phi, msg=msg)
# extract projected parameters from message
x_bd_str = msg[2].split("Found boundary point: ")[1].split(" Residual vector: ")[0]
num_list = [float(val) for val in x_bd_str.replace("[", "").replace("]", "").split(" ")]
alpha_proj_py, gamma_proj_py, phi_proj_py = num_list

proc_eval = subprocess.run(
    ["./../generated_c_code/test_projection", "project", str(alpha), str(gamma), str(phi)],
    capture_output=True, text=True
)
num_list = proc_eval.stdout.split("Projected (alpha, gamma, phi) = ")[1].split("\n")[0].replace("(", "").replace(")", "").split(", ")
alpha_proj_c, gamma_proj_c, phi_proj_c = [float(val) for val in num_list]

# Create a table to compare projected parameters
print(f"{'Parameter':>10}  {'Python Projection':>15}  {'C Projection':>15}  {'Abs Error':>12}")
for name, py_val, c_val in zip(['alpha', 'gamma', 'phi'], [alpha_proj_py, gamma_proj_py, phi_proj_py], [alpha_proj_c, gamma_proj_c, phi_proj_c]):
    abs_err = abs(py_val - c_val)
    print(f"{name:>10}  {py_val:>15.6f}  {c_val:>15.6f}  {abs_err:>12.2e}")
    
print(proc_eval.stderr)
for m_ in msg :
    print(m_)

## Report on the use of the surrogate in TCV 2x2v regression test
Resolution : 8 x 8 x 8 x 8

Time without surrogate                          7.6 seconds     (BC time 1% total time)
Time with surrogate, but without projection     8.0 seconds     (BC time 6% total time)
Time with surrogate and projection             50.0 seconds     (BC time 85% total time)